# 1. Import Libraries

In [14]:
import numpy as np
import tensorflow as tf
import itertools
import sys
import os

# 2. Data & Model Paths

## 1. System paths

In [ ]:
sys.path.append(r'C:/Users/Basudewa/Documents/Mata Kuliah/Ujian Proposal/Programs/Scripts')

## 2. Folders

In [ ]:
data_folder = r'C:/Users/Basudewa/Documents/Mata Kuliah/Ujian Proposal/Programs/Data/TrainTest'

# 3. Import Data & Models

## 1. Import Models

In [8]:
from model import Encoder, Decoder, VAE, RSVD

## 2. Import Data

In [ ]:
train_data_path = os.path.join(data_folder, 'train_data.npy')

try:
    train_data = np.load(train_data_path).astype(np.float32)
    num_items = train_data.shape[1]
    print(f"Data latih berhasil dimuat.")
    print(f"   Dimensi data: {train_data.shape} (Pengguna x Film)")
except FileNotFoundError:
    print("Error: File 'train_data.npy' tidak ditemukan. Pastikan Anda sudah menjalankan preprocessing.")

2. Data latih berhasil dimuat.
   Dimensi data: (943, 1682) (Pengguna x Film)


# 4. Hyperparameter Tuning

## 1.VAE

### 1. Hyperparameter Search Space

In [18]:
vae_param_grid = {
    'latent_dim': [10, 20, 30, 40, 50],             # Menguji ukuran ruang laten
    'learning_rate': [0.001, 0.01],     # Menguji kecepatan belajar optimizer
    'batch_size': [64, 128]             # Menguji jumlah data per iterasi
}

# Membuat semua kemungkinan kombinasi
vae_keys, vae_values = zip(*vae_param_grid.items())
vae_combinations = [dict(zip(vae_keys, v)) for v in itertools.product(*vae_values)]

best_vae_loss = float('inf')
best_vae_params = None
best_vae_model = None

### 2. Grid Search

In [19]:
for i, params in enumerate(vae_combinations):
    print(f"\n[{i+1}/{len(vae_combinations)}] Menguji VAE | Params: {params}")
    
    # Inisialisasi model baru
    encoder = Encoder(hidden_dims=[512, 256], latent_dim=params['latent_dim'])
    decoder = Decoder(hidden_dims=[256, 512], output_dim=num_items)
    vae = VAE(encoder, decoder)
    
    optimizer = tf.keras.optimizers.Adam(learning_rate=params['learning_rate'])
    vae.compile(optimizer=optimizer)
    
    # Latih model (gunakan epochs kecil, misal 10, agar tuning cepat)
    history = vae.fit(train_data, train_data, epochs=10, batch_size=params['batch_size'], verbose=0)
    
    # Ambil Total Loss dari epoch terakhir
    final_loss = history.history['loss'][-1]
    print(f"   -> Final Total Loss: {final_loss:.4f}")
    
    # Cek apakah ini model terbaik sejauh ini
    if final_loss < best_vae_loss:
        best_vae_loss = final_loss
        best_vae_params = params
        best_vae_model = vae

print("\n==================================================")
print(f"[HASIL TERBAIK VAE]")
print(f"Parameter Terbaik: {best_vae_params}")
print(f"Loss Terendah: {best_vae_loss:.4f}")
print("==================================================")


[1/20] Menguji VAE | Params: {'latent_dim': 10, 'learning_rate': 0.001, 'batch_size': 64}


c:\Users\Basudewa\Documents\Mata Kuliah\Ujian Proposal\Programs\venv\lib\site-packages\keras\src\layers\layer.py:1493: UserWarning: Layer 'encoder' looks like it has unbuilt state, but Keras is not able to trace the layer `call()` in order to build it automatically. Possible causes:
1. The `call()` method of your layer may be crashing. Try to `__call__()` the layer eagerly on some test input first to see if it works. E.g. `x = np.random.random((3, 4)); y = layer(x)`
2. If the `call()` method is correct, then you may need to implement the `def build(self, input_shape)` method on your layer. It should create all variables used by the layer (e.g. by calling `layer.build()` on all its children layers).
Exception encountered: ''Layer "dense" expects 1 input(s), but it received 2 input tensors. Inputs received: [<tf.Tensor 'data:0' shape=(None, 1682) dtype=float32>, <tf.Tensor 'data_1:0' shape=(None, 1682) dtype=float32>]''
  warnings.warn(
c:\Users\Basudewa\Documents\Mata Kuliah\Ujian Propo

ValueError: Exception encountered when calling Encoder.call().

[1mLayer "dense" expects 1 input(s), but it received 2 input tensors. Inputs received: [<tf.Tensor 'data:0' shape=(None, 1682) dtype=float32>, <tf.Tensor 'data_1:0' shape=(None, 1682) dtype=float32>][0m

Arguments received by Encoder.call():
  • inputs=('tf.Tensor(shape=(None, 1682), dtype=float32)', 'tf.Tensor(shape=(None, 1682), dtype=float32)')

### 3. Extract Latent Representation

In [ ]:
# Menggunakan VAE terbaik untuk memprediksi nilai rata-rata laten (z_mean)
# Kita gunakan batch_size terbaik yang ditemukan dari tuning VAE
best_batch_size = best_vae_params['batch_size']
z_mean, _ = best_vae_model.encoder.predict(train_data, batch_size=best_batch_size)

# Menyimpan hasil ke dalam variabel latent_matrix_Z
latent_matrix_Z = z_mean

print(f"Ekstraksi berhasil! Dimensi Matriks Laten Z: {latent_matrix_Z.shape}")

## 2. RSVD

### 1. Hyperparameter Search Space

In [ ]:
rsvd_param_grid = {
    'n_factors': [10, 20, 30],       # Dimensi dekomposisi / faktor laten k
    'learning_rate': [0.01, 0.05],   # Learning rate SGD
    'lambda_reg': [0.01, 0.1]        # Penalti regularisasi L2
}

rsvd_keys, rsvd_values = zip(*rsvd_param_grid.items())
rsvd_combinations = [dict(zip(rsvd_keys, v)) for v in itertools.product(*rsvd_values)]

best_rsvd_error = float('inf')
best_rsvd_params = None
best_rsvd_model = None

### 2. Grid Search

In [ ]:
for i, params in enumerate(rsvd_combinations):
    print(f"\n[{i+1}/{len(rsvd_combinations)}] Menguji RSVD | Params: {params}")
    
    # Inisialisasi model RSVD dengan parameter saat ini
    rsvd = RSVD(
        n_factors=params['n_factors'], 
        learning_rate=params['learning_rate'], 
        lambda_reg=params['lambda_reg'], 
        epochs=20  # Epoch kecil untuk tuning
    )
    
    # Latih RSVD menggunakan matriks laten Z
    rsvd.fit(latent_matrix_Z)
    
    # Evaluasi seberapa baik dekomposisi merekonstruksi Z (Menghitung MSE)
    # Z_rekonstruksi = U * Sigma * V^T
    reconstructed_Z = np.dot(np.dot(rsvd.U, rsvd.Sigma), rsvd.V.T)
    mse_error = np.mean(np.square(latent_matrix_Z - reconstructed_Z))
    
    print(f"   -> MSE Rekonstruksi Z: {mse_error:.4f}")
    
    # Cek apakah ini kombinasi terbaik
    if mse_error < best_rsvd_error:
        best_rsvd_error = mse_error
        best_rsvd_params = params
        best_rsvd_model = rsvd

print("\n==================================================")
print(f"[HASIL TERBAIK RSVD]")
print(f"Parameter Terbaik: {best_rsvd_params}")
print(f"MSE Terendah pada Ruang Laten: {best_rsvd_error:.4f}")
print("==================================================")